# Solar PV Thermal Defect Detection (LWIR 8-14µm)
## Exploratory Data Analysis & Class Verification

This notebook demonstrates:
1. **Class Distribution Audit:** Verifying the 10,000 normal vs 10,000 defect samples across 11 anomaly classes.
2. **Thermal Signatures Visual Grid:** Visualizing sample LWIR images under thermal colormaps (`inferno` / `magma`).
3. **Thermal Preprocessing (CLAHE):** Comparing raw thermal images against Contrast Limited Adaptive Histogram Equalization.
4. **PyTorch DataLoader Pipeline:** Verifying batch shapes and tensor statistics.

In [ ]:
import os
import sys
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch

# Add src directory to path
sys.path.append(os.path.abspath("../src"))
from dataset import CLASSES, CLASS_TO_IDX, create_dataloaders, get_thermal_transforms

### 1. Load Metadata and Verify Sample Counts

In [ ]:
data_dir = "../data/infrared_solar_modules"
json_path = os.path.join(data_dir, "module_metadata.json")

class_counts = {cls: 0 for cls in CLASSES}
sample_paths = {cls: [] for cls in CLASSES}

if os.path.exists(json_path):
    with open(json_path, "r") as f:
        meta = json.load(f)
    items = meta.values() if isinstance(meta, dict) else meta
    for item in items:
        rel = item.get("image_filepath", item.get("filename", ""))
        fpath = os.path.join(data_dir, rel) if not os.path.isabs(rel) else rel
        cname = item.get("anomaly_class", item.get("class", "No-Anomaly"))
        if cname in class_counts:
            class_counts[cname] += 1
            if len(sample_paths[cname]) < 5 and os.path.exists(fpath):
                sample_paths[cname].append(fpath)

total = sum(class_counts.values())
print(f"Total Images Indexed: {total:,}")
for c, cnt in class_counts.items():
    pct = (cnt / total * 100) if total > 0 else 0
    print(f"{c:<18}: {cnt:>5} ({pct:>5.1f}%)")

### 2. Plot Class Distribution

In [ ]:
plt.figure(figsize=(12, 5))
bars = plt.bar(CLASSES, [class_counts[c] for c in CLASSES], color="#2563eb", edgecolor="black")
if len(bars) > 0:
    bars[0].set_color("#16a34a")  # Normal baseline in green
plt.xticks(rotation=45, ha="right")
plt.title("Raptor Maps Solar PV Dataset: 12-Class Distribution", fontsize=13, fontweight="bold")
plt.ylabel("Count")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### 3. Visual Grid: Thermal LWIR Signatures (3x4)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for idx, cls in enumerate(CLASSES):
    ax = axes[idx]
    if sample_paths[cls]:
        img = Image.open(sample_paths[cls][0]).convert("L")
        im = ax.imshow(img, cmap="inferno")
        ax.set_title(f"{cls}", fontsize=11, fontweight="bold")
        plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
    else:
        ax.text(0.5, 0.5, "No sample", ha="center", va="center")
        ax.set_title(cls, fontsize=11)
    ax.axis("off")

plt.suptitle("Sample LWIR Thermal Radiance Profiles across Anomaly Types", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

### 4. PyTorch DataLoader Batch Verification

In [ ]:
try:
    train_loader, val_loader, test_loader = create_dataloaders(data_dir=data_dir, batch_size=16)
    sample_imgs, sample_labels = next(iter(train_loader))
    print(f"Batch Image Tensor Shape: {sample_imgs.shape}")
    print(f"Batch Labels Tensor Shape: {sample_labels.shape}")
    print(f"Tensor Min: {sample_imgs.min().item():.3f}, Max: {sample_imgs.max().item():.3f}")
    print(" DataLoader verified successfully!")
except Exception as e:
    print(f"Notice: Place dataset in {data_dir} to run batch verification ({e})")